# 数据质量、幂等性与血缘追踪

**适用人群**: 高级数据工程师面试备考  
**难度**: ⭐⭐⭐⭐  
**高频考点**: 幂等性设计、数据质量框架、Schema 演进、数据血缘

---

## 本节知识图谱

```
数据质量与可靠性
├── 数据质量框架
│   ├── Great Expectations
│   │   ├── Expectation Suite
│   │   ├── Checkpoint
│   │   └── DataDocs
│   └── dbt Tests
│       ├── 内置测试 (unique/not_null/...)
│       └── 自定义通用测试
├── 幂等性设计 (高频!)
│   ├── INSERT OVERWRITE vs APPEND
│   ├── MERGE/UPSERT 模式
│   └── 分区作为重处理单元
├── 数据血缘 Lineage
│   ├── OpenLineage / Marquez
│   ├── 列级血缘
│   └── 事故响应应用
├── Schema Evolution 策略
│   ├── 兼容性规则
│   ├── Schema Registry
│   └── 破坏性 vs 非破坏性变更
└── Backfill 策略
    ├── Airflow backfill 命令
    ├── 风险与缓解
    └── 安全 Backfill 模式
```

---
## 1. Great Expectations 数据质量框架

### 1.1 核心概念

```
Great Expectations 架构:

  Data Source          Expectation Suite          Checkpoint
  (数据源)             (期望集合)                  (执行检查)
     │                      │                          │
     │   ┌──────────────────┤                          │
     │   │ expect_column_values_to_not_be_null()        │
     │   │ expect_column_values_to_be_between()         │
     │   │ expect_column_to_exist()                     │
     │   │ expect_table_row_count_to_be_between()       │
     │   └──────────────────┤                          │
     │                      │                          │
     └──────────────────────┴─────────────────────────>│
                                                        │
                                                   ValidationResult
                                                        │
                                                   ┌────▼────┐
                                                   │DataDocs │
                                                   │(报告网站) │
                                                   └─────────┘
```

| 概念 | 作用 |
|------|------|
| **Expectation** | 单条数据质量断言（如：列不为空） |
| **Expectation Suite** | 一组 Expectations 的集合（如：订单表的所有质量规则） |
| **Batch** | 被验证的数据批次（一个 DataFrame 或 SQL 查询结果） |
| **Checkpoint** | 将数据源与 Suite 绑定的运行配置，可集成到 CI/CD 或 Airflow |
| **DataDocs** | 自动生成的 HTML 质量报告 |
| **ValidationResult** | 每次检查的结果（pass/fail + 统计信息） |

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, date
from typing import Dict, List, Any, Optional
import json

# ===== 模拟 Great Expectations 核心功能 =====
# (不需要安装 great_expectations 包)

class ExpectationResult:
    def __init__(self, expectation_type, success, observed_value, unexpected_count=0):
        self.expectation_type = expectation_type
        self.success = success
        self.observed_value = observed_value
        self.unexpected_count = unexpected_count

    def __repr__(self):
        status = "PASS" if self.success else "FAIL"
        return (f"[{status}] {self.expectation_type} | "
                f"observed={self.observed_value} | "
                f"unexpected={self.unexpected_count}")


class DataValidator:
    """模拟 Great Expectations 的核心验证逻辑"""

    def __init__(self, df: pd.DataFrame):
        self.df = df
        self.results: List[ExpectationResult] = []

    def expect_column_to_exist(self, column: str) -> ExpectationResult:
        success = column in self.df.columns
        result = ExpectationResult(
            f"expect_column_to_exist('{column}')",
            success,
            observed_value=list(self.df.columns)
        )
        self.results.append(result)
        return result

    def expect_column_values_to_not_be_null(self, column: str) -> ExpectationResult:
        if column not in self.df.columns:
            result = ExpectationResult(f"expect_column_values_to_not_be_null('{column}')",
                                       False, "Column not found")
        else:
            null_count = self.df[column].isna().sum()
            result = ExpectationResult(
                f"expect_column_values_to_not_be_null('{column}')",
                success=(null_count == 0),
                observed_value=f"{null_count} nulls out of {len(self.df)}",
                unexpected_count=int(null_count)
            )
        self.results.append(result)
        return result

    def expect_column_values_to_be_between(self, column: str, min_value, max_value) -> ExpectationResult:
        if column not in self.df.columns:
            result = ExpectationResult(
                f"expect_column_values_to_be_between('{column}', {min_value}, {max_value})",
                False, "Column not found")
        else:
            out_of_range = ((self.df[column] < min_value) | (self.df[column] > max_value)).sum()
            result = ExpectationResult(
                f"expect_column_values_to_be_between('{column}', {min_value}, {max_value})",
                success=(out_of_range == 0),
                observed_value=f"range=[{self.df[column].min()}, {self.df[column].max()}]",
                unexpected_count=int(out_of_range)
            )
        self.results.append(result)
        return result

    def expect_column_values_to_be_in_set(self, column: str, value_set: set) -> ExpectationResult:
        if column not in self.df.columns:
            result = ExpectationResult(
                f"expect_column_values_to_be_in_set('{column}')", False, "Column not found")
        else:
            invalid = (~self.df[column].isin(value_set)).sum()
            unique_vals = set(self.df[column].unique())
            result = ExpectationResult(
                f"expect_column_values_to_be_in_set('{column}')",
                success=(invalid == 0),
                observed_value=f"unique_values={unique_vals}",
                unexpected_count=int(invalid)
            )
        self.results.append(result)
        return result

    def expect_column_values_to_be_unique(self, column: str) -> ExpectationResult:
        if column not in self.df.columns:
            result = ExpectationResult(
                f"expect_column_values_to_be_unique('{column}')", False, "Column not found")
        else:
            duplicates = self.df[column].duplicated().sum()
            result = ExpectationResult(
                f"expect_column_values_to_be_unique('{column}')",
                success=(duplicates == 0),
                observed_value=f"{duplicates} duplicates",
                unexpected_count=int(duplicates)
            )
        self.results.append(result)
        return result

    def expect_table_row_count_to_be_between(self, min_value: int, max_value: int) -> ExpectationResult:
        row_count = len(self.df)
        result = ExpectationResult(
            f"expect_table_row_count_to_be_between({min_value}, {max_value})",
            success=(min_value <= row_count <= max_value),
            observed_value=row_count
        )
        self.results.append(result)
        return result

    def validate(self) -> dict:
        """返回汇总验证结果"""
        passed = sum(1 for r in self.results if r.success)
        failed = len(self.results) - passed
        return {
            "success": failed == 0,
            "statistics": {
                "evaluated_expectations": len(self.results),
                "successful_expectations": passed,
                "unsuccessful_expectations": failed,
                "success_percent": round(passed / len(self.results) * 100, 1) if self.results else 0
            },
            "results": self.results
        }


# ===== 创建测试数据 =====
orders_df = pd.DataFrame({
    "order_id":    [1001, 1002, 1003, 1004, 1002],   # 1002 重复
    "user_id":     [1,    2,    None, 4,    2],        # 1003 有空值
    "amount":      [50.0, 200.0, 30.0, -10.0, 150.0], # -10 超出范围
    "status":      ["paid", "pending", "paid", "invalid", "paid"], # invalid 状态
    "created_at":  pd.date_range("2024-01-01", periods=5),
})

print("测试数据集:")
print(orders_df.to_string())

# ===== 运行数据质量检查 =====
print("\n" + "=" * 60)
print("运行 Great Expectations 质量检查")
print("=" * 60)

validator = DataValidator(orders_df)

# 定义 Expectation Suite
validator.expect_column_to_exist("order_id")
validator.expect_column_to_exist("payment_method")  # 不存在的列
validator.expect_column_values_to_not_be_null("order_id")
validator.expect_column_values_to_not_be_null("user_id")  # 有空值
validator.expect_column_values_to_be_unique("order_id")   # 有重复
validator.expect_column_values_to_be_between("amount", 0, 10000)  # 有负值
validator.expect_column_values_to_be_in_set("status", {"paid", "pending", "cancelled"})  # invalid 状态
validator.expect_table_row_count_to_be_between(1, 100)

results = validator.validate()

print(f"\n验证结果: {'通过' if results['success'] else '失败'}")
print(f"统计: {results['statistics']}")
print("\n详细结果:")
for r in results["results"]:
    print(f"  {r}")

In [ ]:
# Great Expectations 真实 API 示例（展示实际使用方式）
ge_real_api_example = '''
import great_expectations as gx
from great_expectations.checkpoint import Checkpoint

# ===== 初始化 Data Context =====
context = gx.get_context()

# ===== 创建数据源 (Pandas) =====
datasource = context.sources.add_pandas("my_pandas_datasource")
data_asset = datasource.add_dataframe_asset("orders_asset")
batch_request = data_asset.build_batch_request(dataframe=orders_df)

# ===== 创建 Expectation Suite =====
suite = context.add_expectation_suite("orders_quality_suite")
validator = context.get_validator(
    batch_request=batch_request,
    expectation_suite_name="orders_quality_suite"
)

# 添加各种 Expectations
validator.expect_column_to_exist("order_id")
validator.expect_column_values_to_not_be_null("order_id")
validator.expect_column_values_to_not_be_null("user_id")
validator.expect_column_values_to_be_unique("order_id")
validator.expect_column_values_to_be_between(
    "amount",
    min_value=0.01,
    max_value=100000.00,
    mostly=0.99  # 允许 1% 例外 (mostly 参数)
)
validator.expect_column_values_to_be_in_set(
    "status",
    value_set=["paid", "pending", "cancelled", "refunded"]
)
validator.expect_column_values_to_match_regex(
    "email",
    regex=r"^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\\.[a-zA-Z]{2,}$"
)
validator.expect_table_row_count_to_be_between(min_value=1000, max_value=1_000_000)

# 保存 Suite
validator.save_expectation_suite(discard_failed_expectations=False)

# ===== 创建并运行 Checkpoint =====
checkpoint = context.add_or_update_checkpoint(
    name="orders_checkpoint",
    validations=[
        {
            "batch_request": batch_request,
            "expectation_suite_name": "orders_quality_suite",
        }
    ],
    action_list=[
        # 自动生成 DataDocs
        {"name": "store_validation_result", "action": {"class_name": "StoreValidationResultAction"}},
        {"name": "update_data_docs",        "action": {"class_name": "UpdateDataDocsAction"}},
        # 失败时发送 Slack 通知
        {
            "name": "send_slack_notification",
            "action": {
                "class_name": "SlackNotificationAction",
                "slack_webhook": "https://hooks.slack.com/...",
                "notify_on": "failure",
            }
        }
    ]
)

# 在 Airflow Task 中调用
result = checkpoint.run()
if not result["success"]:
    raise ValueError(f"数据质量检查失败: {result}")
'''

print("Great Expectations 真实 API 示例（在 Airflow 中集成）:")
print(ge_real_api_example)

---
## 2. dbt 测试框架

### 2.1 dbt Tests 类型

```
dbt Tests 分类:

schema.yml 内置测试          自定义测试
├── unique                   ├── 通用测试 (Generic Tests)
├── not_null                 │   (宏 + schema.yml 调用)
├── accepted_values          └── Singular Tests
└── relationships                (tests/ 目录下的 SQL 文件)
    (外键约束)
```

In [ ]:
# dbt 测试配置示例

dbt_schema_yml = '''
# models/schema.yml
version: 2

models:
  - name: orders
    description: "用户订单事实表"
    columns:
      - name: order_id
        description: "订单唯一 ID"
        tests:
          - unique                      # 内置: 唯一性
          - not_null                    # 内置: 非空

      - name: user_id
        tests:
          - not_null
          - relationships:              # 内置: 外键约束
              to: ref('users')          # 引用 users 模型
              field: user_id

      - name: status
        tests:
          - not_null
          - accepted_values:            # 内置: 枚举值
              values: ['paid', 'pending', 'cancelled', 'refunded']
              quote: true

      - name: amount
        tests:
          - not_null
          - dbt_utils.accepted_range:   # dbt-utils 包的测试
              min_value: 0
              max_value: 100000
              inclusive: true

      - name: created_at
        tests:
          - not_null
          - dbt_utils.not_null_proportion:  # 允许 0.1% 的空值
              at_least: 0.999

    # 模型级别测试 (表级别)
    tests:
      - dbt_utils.expression_is_true:
          expression: "amount >= 0 OR status = 'refunded'"
      - dbt_utils.recency:
          datepart: day
          field: created_at
          interval: 1      # 确保数据在过去 1 天内有更新
'''

dbt_custom_generic_test = '''
# tests/generic/is_valid_email.sql
# 自定义通用测试: 验证邮箱格式
{% test is_valid_email(model, column_name) %}

SELECT {{ column_name }}
FROM {{ model }}
WHERE {{ column_name }} IS NOT NULL
  AND NOT REGEXP_CONTAINS(
    {{ column_name }},
    r"^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\\.[a-zA-Z]{2,}$"
  )

{% endtest %}

# 在 schema.yml 中调用:
# - is_valid_email  # tests/generic/is_valid_email.sql 中的宏
'''

dbt_singular_test = '''
# tests/assert_orders_total_matches_payments.sql
# Singular Test: 验证订单总额与支付总额一致
-- 这个测试应该返回 0 行（有结果 = 测试失败）

WITH order_totals AS (
    SELECT
        DATE(created_at) AS order_date,
        SUM(amount)      AS total_amount
    FROM {{ ref('orders') }}
    WHERE status = 'paid'
    GROUP BY 1
),
payment_totals AS (
    SELECT
        DATE(processed_at) AS payment_date,
        SUM(amount)        AS total_amount
    FROM {{ ref('payments') }}
    WHERE status = 'completed'
    GROUP BY 1
)
SELECT
    o.order_date,
    o.total_amount AS order_total,
    p.total_amount AS payment_total,
    ABS(o.total_amount - p.total_amount) AS discrepancy
FROM order_totals o
JOIN payment_totals p ON o.order_date = p.payment_date
WHERE ABS(o.total_amount - p.total_amount) > 0.01  -- 允许 1 分钱的浮点误差
'''

dbt_store_failures = '''
# dbt_project.yml: 全局开启 store_failures
# 这会把测试失败的具体行存入数据库，方便调试
tests:
  +store_failures: true
  +schema: dbt_test_failures   # 存入专门的 schema

# 单独为某个测试开启:
# - unique:
#     config:
#       store_failures: true
#       limit: 100  # 只存前 100 行失败数据
'''

print("dbt schema.yml 内置测试:")
print(dbt_schema_yml)
print("\ndbt 自定义通用测试 (Generic Test):")
print(dbt_custom_generic_test)
print("\ndbt Singular Test (验证业务逻辑):")
print(dbt_singular_test)

---
## 3. 幂等性设计 (Idempotency)

### 3.1 什么是幂等性？

> **幂等性**: 对同一操作执行多次，结果与执行一次相同。

**为什么数据管道必须幂等？**
- 任务失败后需要**安全重试**
- Backfill（历史补跑）需要**覆盖旧数据**
- 避免数据重复（Double-counting）

### 3.2 幂等性模式对比

```
不幂等的写入模式:
┌─────────────────────────────────────────────┐
│ INSERT APPEND (每次运行都追加)               │
│                                             │
│  第1次运行: [A, B, C]                       │
│  第2次运行: [A, B, C, A, B, C]  ← 数据重复! │
└─────────────────────────────────────────────┘

幂等的写入模式:
┌─────────────────────────────────────────────┐
│ INSERT OVERWRITE (覆盖写)                   │
│                                             │
│  第1次运行: [A, B, C]                       │
│  第2次运行: [A, B, C]  ← 结果相同!          │
└─────────────────────────────────────────────┘

┌─────────────────────────────────────────────┐
│ MERGE / UPSERT (合并更新)                   │
│                                             │
│  已有: {id:1, name:"Alice"}                 │
│  新数据: {id:1, name:"Alicia"}, {id:2, ...} │
│  结果: {id:1, name:"Alicia"}, {id:2, ...}   │
│         ↑ 更新而非重复插入                   │
└─────────────────────────────────────────────┘
```

In [ ]:
import pandas as pd
from datetime import date, datetime

# ===== 模拟幂等 ETL 的各种模式 =====

class IdempotentETLSimulator:
    """模拟幂等 ETL 的不同写入模式"""

    def __init__(self):
        # 模拟目标数据库中的数据
        self.warehouse: Dict[str, pd.DataFrame] = {}

    def get_table(self, table_name: str) -> pd.DataFrame:
        return self.warehouse.get(table_name, pd.DataFrame())

    # 模式 1: INSERT APPEND (非幂等，不推荐)
    def append_insert(self, table_name: str, new_data: pd.DataFrame):
        """非幂等: 每次运行都追加数据"""
        existing = self.get_table(table_name)
        self.warehouse[table_name] = pd.concat([existing, new_data], ignore_index=True)

    # 模式 2: INSERT OVERWRITE (幂等)
    def overwrite_partition(self, table_name: str, partition_col: str,
                            partition_value: Any, new_data: pd.DataFrame):
        """幂等: 覆盖特定分区，其余分区不变"""
        existing = self.get_table(table_name)
        if not existing.empty:
            # 删除目标分区的旧数据
            existing = existing[existing[partition_col] != partition_value]
        # 写入新数据
        self.warehouse[table_name] = pd.concat([existing, new_data], ignore_index=True)

    # 模式 3: MERGE/UPSERT (幂等)
    def merge_upsert(self, table_name: str, new_data: pd.DataFrame,
                     key_cols: List[str], update_cols: List[str]):
        """幂等: 存在则更新，不存在则插入"""
        existing = self.get_table(table_name)
        if existing.empty:
            self.warehouse[table_name] = new_data.copy()
            return

        # 合并数据
        merged = existing.merge(
            new_data[key_cols + update_cols],
            on=key_cols,
            how="outer",
            suffixes=("_old", "_new")
        )

        # 用新值覆盖旧值
        for col in update_cols:
            if f"{col}_new" in merged.columns:
                merged[col] = merged[f"{col}_new"].combine_first(merged.get(f"{col}_old", pd.Series()))
                merged.drop(columns=[f"{col}_old", f"{col}_new"], inplace=True, errors="ignore")

        self.warehouse[table_name] = merged

    # 模式 4: DELETE + INSERT (幂等，原子性差)
    def delete_then_insert(self, table_name: str, partition_col: str,
                           partition_value: Any, new_data: pd.DataFrame):
        """幂等: 先删除再插入（简单但非原子）"""
        existing = self.get_table(table_name)
        if not existing.empty:
            existing = existing[existing[partition_col] != partition_value]
        self.warehouse[table_name] = pd.concat([existing, new_data], ignore_index=True)


# ===== 演示非幂等 vs 幂等 =====
print("=" * 60)
print("演示 1: APPEND (非幂等) - 重复运行导致数据重复")
print("=" * 60)

sim = IdempotentETLSimulator()
batch = pd.DataFrame({
    "date":     ["2024-01-01"] * 3,
    "user_id":  [1, 2, 3],
    "revenue":  [100, 200, 150],
})

print("\n第 1 次运行 (正常):")
sim.append_insert("daily_revenue", batch)
print(sim.get_table("daily_revenue").to_string(index=False))

print("\n第 2 次运行 (重跑/重试):")
sim.append_insert("daily_revenue", batch)
df = sim.get_table("daily_revenue")
print(df.to_string(index=False))
print(f"行数: {len(df)} (应该是 3，但现在是 {len(df)}！数据重复了！)")

print("\n" + "=" * 60)
print("演示 2: INSERT OVERWRITE 分区 (幂等) - 重复运行结果相同")
print("=" * 60)

sim2 = IdempotentETLSimulator()
# 先写入 2024-01-01 的数据
sim2.overwrite_partition("daily_revenue", "date", "2024-01-01", batch)
# 再写入 2024-01-02 的数据
batch2 = pd.DataFrame({"date": ["2024-01-02"] * 2, "user_id": [4, 5], "revenue": [300, 250]})
sim2.overwrite_partition("daily_revenue", "date", "2024-01-02", batch2)

print("\n初始状态 (两天数据):")
print(sim2.get_table("daily_revenue").to_string(index=False))

print("\n重跑 2024-01-01 (模拟任务失败重试):")
sim2.overwrite_partition("daily_revenue", "date", "2024-01-01", batch)
df2 = sim2.get_table("daily_revenue")
print(df2.to_string(index=False))
print(f"行数: {len(df2)} (保持正确的 5 行，2024-01-02 数据未受影响)")

In [ ]:
# 展示 SQL 层面的幂等性实现
idempotent_sql_patterns = {
    "BigQuery: INSERT OVERWRITE 分区": '''
-- 覆盖写入单个分区（幂等）
-- 可以安全重跑，不会产生重复数据
INSERT OVERWRITE TABLE sales.daily_orders
PARTITION (dt = '{{ ds }}')
SELECT
    order_id,
    user_id,
    amount,
    status
FROM staging.raw_orders
WHERE DATE(created_at) = '{{ ds }}';
''',
    "Snowflake/BigQuery: MERGE (UPSERT)": '''
-- MERGE 语句 (幂等，支持更新已有记录)
MERGE INTO target_table AS target
USING source_table AS source
    ON target.order_id = source.order_id   -- 匹配条件
WHEN MATCHED AND source.updated_at > target.updated_at THEN
    UPDATE SET
        target.status     = source.status,
        target.amount     = source.amount,
        target.updated_at = source.updated_at
WHEN NOT MATCHED THEN
    INSERT (order_id, user_id, amount, status, created_at)
    VALUES (source.order_id, source.user_id, source.amount,
            source.status, source.created_at);
''',
    "Python: 幂等 ETL 函数": '''
def idempotent_etl(execution_date: str, dry_run: bool = False):
    """
    幂等 ETL 函数设计原则:
    1. 以 execution_date (分区) 为处理单元
    2. 先清除目标分区，再写入新数据
    3. 支持 dry_run 模式（不实际写入）
    """
    print(f"处理日期分区: {execution_date}")

    # Step 1: 提取
    df = extract_from_source(date=execution_date)
    print(f"提取 {len(df)} 行")

    # Step 2: 转换
    df = transform(df)

    # Step 3: 幂等写入
    if not dry_run:
        # 关键: 先删除该分区的旧数据
        delete_partition(table="daily_orders", partition_date=execution_date)
        # 再写入新数据
        write_to_warehouse(df, table="daily_orders", partition_date=execution_date)
        print(f"成功写入 {len(df)} 行到分区 {execution_date}")
    else:
        print(f"[DRY RUN] 将写入 {len(df)} 行")

    return {"rows": len(df), "date": execution_date, "status": "success"}
'''
}

for pattern_name, sql in idempotent_sql_patterns.items():
    print(f"\n{'='*60}")
    print(f"模式: {pattern_name}")
    print(sql)

---
## 4. 数据血缘 (Data Lineage) 追踪

### 4.1 为什么需要数据血缘？

```
数据血缘的核心价值:

事故场景: 生产报表数据异常!

WITHOUT 血缘:                    WITH 血缘:
"哪个上游出了问题？"              马上定位:
↓ 手动检查每个管道                raw.orders
↓ 开会讨论                         └─> staging.orders (ETL)
↓ 2-4 小时后找到根因                     └─> dw.fact_orders (dbt)
                                               └─> rpt.daily_sales (报表)
                                    ← 从报表逆向追溯，5 分钟定位!
```

### 4.2 OpenLineage / Marquez

```
OpenLineage 架构:

  Airflow   dbt   Spark   Flink     ← 数据工具 (Emitter)
     │        │     │       │
     └────────┴─────┴───────┘
              │ OpenLineage 标准事件
              ▼
         Marquez / Atlan / DataHub  ← 血缘存储/展示平台
              │
              ▼
     可视化血缘图 + 影响分析
```

### 4.3 血缘粒度

| 粒度 | 描述 | 用途 |
|------|------|------|
| **Job 级** | DAG/Pipeline 之间的依赖 | 了解整体数据流 |
| **Dataset 级** | 表/文件之间的依赖 | 影响分析 |
| **Column 级** | 字段之间的转换关系 | GDPR 合规、字段级影响分析 |

In [ ]:
# 用 Python 模拟数据血缘图（不需要 networkx 外部库）

class LineageGraph:
    """简单的数据血缘追踪器"""

    def __init__(self):
        self.nodes: Dict[str, dict] = {}    # dataset_name -> metadata
        self.edges: List[tuple] = []         # (source, target, job_name)

    def add_dataset(self, name: str, dataset_type: str, **metadata):
        self.nodes[name] = {"type": dataset_type, **metadata}

    def add_lineage(self, source: str, target: str, job: str, column_mapping: dict = None):
        """添加血缘关系"""
        self.edges.append({
            "source": source,
            "target": target,
            "job": job,
            "column_mapping": column_mapping or {}
        })

    def get_upstream(self, dataset: str, depth: int = 3) -> List[str]:
        """获取上游血缘 (影响来源)"""
        result = []
        to_check = [dataset]
        visited = set()
        current_depth = 0

        while to_check and current_depth < depth:
            next_check = []
            for node in to_check:
                upstream = [e["source"] for e in self.edges if e["target"] == node]
                for up in upstream:
                    if up not in visited:
                        result.append(up)
                        next_check.append(up)
                        visited.add(up)
            to_check = next_check
            current_depth += 1
        return result

    def get_downstream(self, dataset: str, depth: int = 3) -> List[str]:
        """获取下游血缘 (影响分析)"""
        result = []
        to_check = [dataset]
        visited = set()
        current_depth = 0

        while to_check and current_depth < depth:
            next_check = []
            for node in to_check:
                downstream = [e["target"] for e in self.edges if e["source"] == node]
                for down in downstream:
                    if down not in visited:
                        result.append(down)
                        next_check.append(down)
                        visited.add(down)
            to_check = next_check
            current_depth += 1
        return result

    def print_graph(self):
        """用 ASCII 打印血缘图"""
        print("数据血缘图 (ASCII):")
        print()
        # 找出所有根节点 (没有上游的)
        all_targets = {e["target"] for e in self.edges}
        roots = [n for n in self.nodes if n not in all_targets]

        def print_tree(node, prefix="", is_last=True):
            connector = "└── " if is_last else "├── "
            node_info = self.nodes.get(node, {})
            type_label = f" [{node_info.get('type', 'unknown')}]"
            print(f"{prefix}{connector}{node}{type_label}")
            children = [e["target"] for e in self.edges if e["source"] == node]
            for i, child in enumerate(children):
                new_prefix = prefix + ("    " if is_last else "│   ")
                print_tree(child, new_prefix, is_last=(i == len(children) - 1))

        for root in roots:
            print_tree(root)


# ===== 构建示例血缘图 =====
lineage = LineageGraph()

# 添加数据集节点
lineage.add_dataset("raw.app_events",         "source",  layer="raw")
lineage.add_dataset("raw.user_profiles",       "source",  layer="raw")
lineage.add_dataset("raw.orders",              "source",  layer="raw")
lineage.add_dataset("staging.user_events",     "staging", layer="staging")
lineage.add_dataset("staging.users",           "staging", layer="staging")
lineage.add_dataset("dw.fact_orders",          "dw",      layer="warehouse")
lineage.add_dataset("dw.dim_users",            "dw",      layer="warehouse")
lineage.add_dataset("rpt.daily_sales",         "report",  layer="reporting")
lineage.add_dataset("rpt.user_cohort_report",  "report",  layer="reporting")
lineage.add_dataset("dashboard.sales_kpi",     "dashboard", layer="viz")

# 添加血缘关系
lineage.add_lineage("raw.app_events",    "staging.user_events", "airflow:etl_events")
lineage.add_lineage("raw.user_profiles", "staging.users",       "airflow:etl_users")
lineage.add_lineage("raw.orders",        "dw.fact_orders",      "dbt:orders_model",
                    column_mapping={"raw.order_id": "dw.order_sk"})
lineage.add_lineage("staging.users",     "dw.dim_users",        "dbt:dim_users")
lineage.add_lineage("staging.user_events", "dw.fact_orders",    "dbt:enrich_orders")
lineage.add_lineage("dw.fact_orders",    "rpt.daily_sales",     "dbt:daily_sales")
lineage.add_lineage("dw.fact_orders",    "rpt.user_cohort_report", "dbt:cohort")
lineage.add_lineage("dw.dim_users",      "rpt.user_cohort_report", "dbt:cohort")
lineage.add_lineage("rpt.daily_sales",   "dashboard.sales_kpi", "metabase:sync")

# 打印血缘图
lineage.print_graph()

# 模拟事故场景
print("\n" + "=" * 60)
print("事故场景: dashboard.sales_kpi 数据异常，快速定位根因")
print("=" * 60)
upstream = lineage.get_upstream("dashboard.sales_kpi", depth=5)
print(f"\n dashboard.sales_kpi 的所有上游数据集:")
for ds in upstream:
    node_info = lineage.nodes.get(ds, {})
    print(f"  -> {ds} [{node_info.get('layer', '?')}]")

print("\n" + "=" * 60)
print("影响分析: raw.user_profiles 质量问题，哪些下游受影响?")
print("=" * 60)
downstream = lineage.get_downstream("raw.user_profiles", depth=5)
print(f"\n raw.user_profiles 的所有下游数据集:")
for ds in downstream:
    node_info = lineage.nodes.get(ds, {})
    print(f"  -> {ds} [{node_info.get('layer', '?')}]")

---
## 5. Schema Evolution 策略

### 5.1 Schema 变更类型

```
Schema 变更兼容性:

非破坏性变更 (向后兼容, 安全):
  + 新增可为 NULL 的列           → 旧代码不受影响
  + 新增有默认值的列             → 旧数据自动填充默认值
  + 扩大数据类型 (INT → BIGINT)  → 旧数据仍可存储

破坏性变更 (不兼容, 危险!):
  ✗ 重命名列                    → 下游 SELECT col 报错
  ✗ 删除列                      → 下游引用该列报错
  ✗ 缩小数据类型 (BIGINT→INT)   → 数据截断/溢出
  ✗ 修改列语义 (USD→CNY)        → 数据计算错误(最危险!)
```

### 5.2 Schema Registry (Confluent/AWS Glue)

```
Schema Registry 工作流:

Producer                Schema Registry           Consumer
   │                          │                      │
   │── 注册新 Schema ─────────>│                      │
   │<── Schema ID (如: 42) ───│                      │
   │                          │                      │
   │── 发送消息(含 Schema ID)──────────────────────>  │
   │                          │                      │
   │                          │<── 获取 Schema #42 ──│
   │                          │── Schema 定义 ──────>│
   │                          │                      │
   │                          │              反序列化消息
```

### 5.3 Avro Schema Evolution 兼容性规则

| 兼容性类型 | 含义 | 规则 |
|-----------|------|------|
| **BACKWARD** | 新 Schema 可读旧数据 | 只能删除或添加默认值字段 |
| **FORWARD** | 旧 Schema 可读新数据 | 只能添加或删除默认值字段 |
| **FULL** | 双向兼容 | 只能添加/删除有默认值的字段 |
| **NONE** | 无兼容性保证 | 任何变更都允许（危险）|

In [ ]:
# Schema Evolution 策略的 Python 模拟

# ===== Avro Schema 演进示例 =====
schema_v1 = {
    "type": "record",
    "name": "Order",
    "namespace": "com.company.events",
    "fields": [
        {"name": "order_id",   "type": "string"},
        {"name": "user_id",    "type": "string"},
        {"name": "amount",     "type": "double"},
        {"name": "status",     "type": "string"},
    ]
}

schema_v2_backward_compatible = {
    "type": "record",
    "name": "Order",
    "namespace": "com.company.events",
    "fields": [
        {"name": "order_id",      "type": "string"},
        {"name": "user_id",       "type": "string"},
        {"name": "amount",        "type": "double"},
        {"name": "status",        "type": "string"},
        # 新增: 有默认值 → 向后兼容
        {"name": "currency",      "type": "string", "default": "USD"},
        # 新增: nullable → 向后兼容
        {"name": "discount_code", "type": ["null", "string"], "default": None},
    ]
}

schema_v3_breaking = {
    "type": "record",
    "name": "Order",
    "namespace": "com.company.events",
    "fields": [
        {"name": "order_id",    "type": "string"},
        {"name": "customer_id", "type": "string"},  # 重命名 user_id → 破坏性!
        {"name": "total",       "type": "double"},   # 重命名 amount → 破坏性!
        {"name": "state",       "type": "string"},   # 重命名 status → 破坏性!
    ]
}

def check_schema_compatibility(old_schema: dict, new_schema: dict) -> dict:
    """简化的 Schema 兼容性检查"""
    old_fields = {f["name"]: f for f in old_schema["fields"]}
    new_fields = {f["name"]: f for f in new_schema["fields"]}

    issues = []
    warnings = []

    # 检查删除的字段
    for name, field in old_fields.items():
        if name not in new_fields:
            issues.append(f"BREAKING: 字段 '{name}' 被删除（下游可能引用此字段）")

    # 检查新增的字段
    for name, field in new_fields.items():
        if name not in old_fields:
            has_default = "default" in field
            is_nullable = isinstance(field["type"], list) and "null" in field["type"]
            if has_default or is_nullable:
                warnings.append(f"OK: 新增字段 '{name}' (有默认值/可为空，向后兼容)")
            else:
                issues.append(f"BREAKING: 新增必填字段 '{name}' (没有默认值，旧数据无法填充)")

    # 检查类型变更
    for name in set(old_fields) & set(new_fields):
        if old_fields[name]["type"] != new_fields[name]["type"]:
            issues.append(f"BREAKING: 字段 '{name}' 类型变更 "
                          f"{old_fields[name]['type']} → {new_fields[name]['type']}")

    return {
        "compatible": len(issues) == 0,
        "issues": issues,
        "warnings": warnings
    }


print("Schema 兼容性检查")
print("=" * 60)

print("\n检查 V1 → V2 (向后兼容变更):")
result = check_schema_compatibility(schema_v1, schema_v2_backward_compatible)
print(f"兼容性: {'通过' if result['compatible'] else '不通过'}")
for w in result["warnings"]: print(f"  ✓ {w}")
for i in result["issues"]:   print(f"  ✗ {i}")

print("\n检查 V1 → V3 (破坏性变更):")
result = check_schema_compatibility(schema_v1, schema_v3_breaking)
print(f"兼容性: {'通过' if result['compatible'] else '不通过'}")
for w in result["warnings"]: print(f"  ✓ {w}")
for i in result["issues"]:   print(f"  ✗ {i}")

# 处理破坏性变更的最佳实践
print("\n" + "=" * 60)
print("处理破坏性 Schema 变更的正确流程:")
steps = [
    "1. 新增新列 (新名称)，保留旧列",
    "2. 更新写入端，同时写入新旧两列",
    "3. 逐步迁移消费端，使用新列",
    "4. 确认所有消费端已迁移",
    "5. 停止写入旧列",
    "6. （可选）在下一个大版本中删除旧列",
]
for step in steps:
    print(f"  {step}")

---
## 6. Backfill 策略与风险

### 6.1 什么是 Backfill？

> **Backfill（历史补跑）**: 对过去某个时间范围重新运行 DAG，通常用于修复历史数据 Bug 或初始化新表。

### 6.2 Backfill 风险矩阵

```
Backfill 主要风险:

┌─────────────────────┬──────────────────────────┬───────────────────────┐
│ 风险                 │ 场景                      │ 缓解措施              │
├─────────────────────┼──────────────────────────┼───────────────────────┤
│ 数据重复             │ 非幂等写入 + 重跑          │ 确保幂等性            │
│ API 限流            │ 大量历史请求触发 rate limit │ 设置并发限制          │
│ 资源竞争            │ Backfill 与日常任务争资源   │ 使用 Pool 隔离        │
│ 竞态条件            │ 并发 DAG Run 同时写入分区   │ max_active_runs=1     │
│ 下游触发            │ 补跑触发 ExternalTaskSensor │ 暂停下游 DAG          │
│ 成本超支            │ 大量历史扫描费用            │ 评估成本再执行        │
└─────────────────────┴──────────────────────────┴───────────────────────┘
```

In [ ]:
# Backfill 命令和最佳实践
backfill_commands = '''
# ===== Airflow Backfill 命令 =====

# 基本补跑命令
airflow dags backfill \\
    --dag-id etl_user_events \\
    --start-date 2024-01-01 \\
    --end-date 2024-01-31

# 安全补跑: 先 dry-run 看会触发哪些 DAG Run
airflow dags backfill \\
    --dag-id etl_user_events \\
    --start-date 2024-01-01 \\
    --end-date 2024-01-31 \\
    --dry-run                     # 只打印，不实际运行!

# 限制并发，避免资源竞争
airflow dags backfill \\
    --dag-id etl_user_events \\
    --start-date 2024-01-01 \\
    --end-date 2024-01-31 \\
    --max-jobs 2                  # 最多同时跑 2 个 DAG Run

# 跳过已成功的 DAG Run（增量补跑）
airflow dags backfill \\
    --dag-id etl_user_events \\
    --start-date 2024-01-01 \\
    --end-date 2024-01-31 \\
    --rerun-failed-tasks          # 只重跑失败的任务

# 重置并重跑特定日期
airflow tasks clear \\
    --dag-id etl_user_events \\
    --task-id transform \\
    --start-date 2024-01-15 \\
    --end-date 2024-01-20 \\
    --yes                         # 不需要交互确认
'''

print("Airflow Backfill 命令示例:")
print(backfill_commands)

# 安全 Backfill 流程
print("\n" + "=" * 60)
print("安全 Backfill 操作流程 (SOP)")
print("=" * 60)

sop_steps = [
    {
        "步骤": "1. 评估影响范围",
        "操作": "确认补跑时间范围、涉及的 DAG/Task、下游依赖",
        "命令": "airflow dags backfill --dry-run ..."
    },
    {
        "步骤": "2. 暂停下游 DAG",
        "操作": "避免 ExternalTaskSensor 触发下游，防止读取到中间状态数据",
        "命令": "airflow dags pause downstream_reporting_dag"
    },
    {
        "步骤": "3. 验证幂等性",
        "操作": "确认 ETL 逻辑使用 INSERT OVERWRITE 或 MERGE，不会产生重复数据",
        "命令": "review ETL code + run on staging environment"
    },
    {
        "步骤": "4. 配置资源池",
        "操作": "为 Backfill 任务使用专用 Pool，避免与日常任务争资源",
        "命令": "airflow pools set backfill_pool 4 'Reserved for backfills'"
    },
    {
        "步骤": "5. 分批执行",
        "操作": "不要一次性补跑 6 个月的数据，分批执行（每次 1 周）",
        "命令": "airflow dags backfill --max-jobs 2 ..."
    },
    {
        "步骤": "6. 监控执行",
        "操作": "实时监控 Airflow UI、资源使用、数据质量检查",
        "命令": "airflow tasks states-for-dag-run ..."
    },
    {
        "步骤": "7. 验证结果",
        "操作": "对比补跑前后的数据指标，确认数据正确",
        "命令": "run data quality checks + compare aggregated metrics"
    },
    {
        "步骤": "8. 恢复下游 DAG",
        "操作": "补跑完成后，重新开启下游 DAG",
        "命令": "airflow dags unpause downstream_reporting_dag"
    },
]

for step in sop_steps:
    print(f"\n[{step['步骤']}]")
    print(f"  操作: {step['操作']}")
    print(f"  命令: {step['命令']}")

---
## 复习要点

### 高频面试问题与答案要点

**Q1: 什么是幂等性？如何设计幂等 ETL？**
> 幂等性指多次执行与一次执行结果相同。实现方式：
> 1. **INSERT OVERWRITE** 指定分区（按 execution_date 分区作为处理单元）
> 2. **MERGE/UPSERT**（根据业务主键更新或插入）
> 3. **DELETE + INSERT**（先删除目标范围，再插入）
> 关键：避免 INSERT APPEND，不要依赖去重逻辑（因为去重本身可能不幂等）。

**Q2: Great Expectations 和 dbt Tests 如何选择？**
> - **dbt Tests**: 适合在数据转换过程中验证，与 dbt 模型紧耦合，CI/CD 集成简单
> - **Great Expectations**: 适合原始数据（摄取阶段）验证，支持更复杂的统计规则，生成 DataDocs 报告，可独立于 dbt 使用
> - 实践中常**结合使用**：GE 验证 raw 层，dbt Tests 验证 staging/mart 层

**Q3: 什么是列级数据血缘？有什么实际用途？**
> 列级血缘追踪字段的转换路径（如 `revenue` 字段从 `raw.amount * exchange_rate` 计算而来）。用途：
> 1. **GDPR 合规**：追踪个人数据在哪里使用
> 2. **字段变更影响分析**：修改某字段逻辑前，了解影响的下游报表
> 3. **数据质量根因分析**：报表数值异常，快速定位是哪个字段的计算逻辑有问题

**Q4: Schema 变更中，哪些是安全的，哪些是危险的？**
> **安全（向后兼容）**: 新增可为 NULL 的列、新增有默认值的列、扩大数据类型（INT→BIGINT）
> **危险（破坏性）**: 重命名列、删除列、缩小数据类型、修改列的业务语义
> 处理破坏性变更：「扩展-迁移-收缩」三步法，新旧并存过渡期

**Q5: Backfill 有哪些风险，如何安全执行？**
> 风险：数据重复（非幂等）、API 限流、资源竞争、竞态条件、下游意外触发
> 安全措施：
> 1. 先 `--dry-run` 确认范围
> 2. 暂停下游 DAG
> 3. 确认 ETL 幂等性
> 4. 使用 `--max-jobs 2` 限制并发
> 5. 分批执行，监控资源

### 关键设计原则

```
数据质量工程黄金法则:

1. 左移质量检查 (Shift Left):
   越早发现问题越好，成本越低
   raw → staging → dw → reporting
   [GE] → [dbt tests] → [monitors] → [alerts]

2. 幂等性第一:
   每个 Task 必须可安全重试
   以分区/主键为处理单元

3. 血缘即文档:
   自动追踪血缘比手动维护文档可靠
   OpenLineage + Marquez/DataHub

4. Schema 只增不改:
   永远不要删除或重命名字段
   新增字段必须有默认值或可为 NULL
```

---
## 练习

### 练习 1: 设计数据质量 Expectation Suite

给定以下用户表结构，编写完整的 Great Expectations Expectation Suite（用模拟代码）：

```
users 表:
- user_id: bigint, 主键
- email: varchar, 唯一, 格式必须合法
- age: int, 范围 13-120
- country: varchar, 只能是 ISO 2位国家代码
- created_at: timestamp, 不能早于 2020-01-01
- subscription_tier: varchar, 值为 free/basic/premium
- monthly_spend: float, 非负数, free 用户必须为 0
```

In [ ]:
# 练习 1 答案
import pandas as pd

# 创建测试数据
users_df = pd.DataFrame({
    "user_id":           [1, 2, 3, 4, 5],
    "email":             ["alice@example.com", "bob@test.org", "invalid-email", "carol@test.com", None],
    "age":               [25, 150, 30, 17, 45],    # 150 超范围
    "country":           ["US", "GB", "CHINA", "CA", "DE"],  # CHINA 不是 2 位码
    "subscription_tier": ["free", "premium", "basic", "vip", "free"],  # vip 不合法
    "monthly_spend":     [0.0, 29.99, 9.99, 49.99, 5.0],  # 最后一个 free 用户 spend != 0
})

validator = DataValidator(users_df)

# 定义 Expectation Suite
validator.expect_column_to_exist("user_id")
validator.expect_column_to_exist("email")
validator.expect_column_to_exist("age")
validator.expect_column_to_exist("country")
validator.expect_column_to_exist("subscription_tier")
validator.expect_column_to_exist("monthly_spend")

validator.expect_column_values_to_not_be_null("user_id")
validator.expect_column_values_to_be_unique("user_id")
validator.expect_column_values_to_not_be_null("email")
validator.expect_column_values_to_be_unique("email")
validator.expect_column_values_to_be_between("age", 13, 120)

# country 检查：2 位 ISO 代码
valid_countries = {"US", "GB", "CA", "DE", "FR", "JP", "CN", "AU"}  # 简化版
validator.expect_column_values_to_be_in_set("country", valid_countries)
validator.expect_column_values_to_be_in_set(
    "subscription_tier", {"free", "basic", "premium"}
)
validator.expect_column_values_to_be_between("monthly_spend", 0, 1000)
validator.expect_table_row_count_to_be_between(1, 100_000_000)

# 业务规则: free 用户 monthly_spend 必须为 0
free_users = users_df[users_df["subscription_tier"] == "free"]
non_zero_free = (free_users["monthly_spend"] != 0).sum()
print(f"业务规则检查 - free 用户 monthly_spend != 0: {non_zero_free} 条违规")

results = validator.validate()
print(f"\n验证结果: {'通过' if results['success'] else '失败'}")
print(f"统计: {results['statistics']}")
print("\n详细结果:")
for r in results["results"]:
    print(f"  {r}")

### 练习 2: 幂等 ETL 实现

以下 ETL 代码存在幂等性问题，修复它：

```python
def daily_sales_etl(execution_date: str):
    # 从 API 提取当天销售数据
    sales = fetch_sales_api(execution_date)

    # 聚合
    summary = aggregate_by_product(sales)

    # 写入数据库（问题在这里）
    db.execute("""
        INSERT INTO daily_product_sales (date, product_id, total_revenue, order_count)
        VALUES (%(date)s, %(product_id)s, %(revenue)s, %(orders)s)
    """, summary)
```

In [ ]:
# 练习 2 答案: 修复幂等性问题

# 模拟数据库和 API
class MockDatabase:
    def __init__(self):
        self.tables = {"daily_product_sales": []}
        self.run_count = 0

    def append_insert(self, date, data):
        """非幂等: 直接追加"""
        self.tables["daily_product_sales"].extend(data)

    def idempotent_write(self, date, data):
        """幂等: 先删除该日期的数据，再插入"""
        # DELETE WHERE date = ?
        self.tables["daily_product_sales"] = [
            row for row in self.tables["daily_product_sales"]
            if row["date"] != date
        ]
        # INSERT
        self.tables["daily_product_sales"].extend(data)

    def count(self, date=None):
        if date:
            return sum(1 for r in self.tables["daily_product_sales"] if r["date"] == date)
        return len(self.tables["daily_product_sales"])


def fetch_sales_mock(date):
    """模拟 API 返回的销售数据"""
    return [
        {"date": date, "product_id": "P001", "total_revenue": 1500.0, "order_count": 30},
        {"date": date, "product_id": "P002", "total_revenue": 800.0,  "order_count": 16},
        {"date": date, "product_id": "P003", "total_revenue": 250.0,  "order_count": 5},
    ]


# ===== 演示非幂等 vs 幂等 =====
print("=" * 60)
print("非幂等 ETL (INSERT APPEND)")
print("=" * 60)

db_bad = MockDatabase()
data = fetch_sales_mock("2024-01-15")

for run_num in range(1, 4):
    db_bad.append_insert("2024-01-15", data)
    count = db_bad.count("2024-01-15")
    print(f"第 {run_num} 次运行后, 2024-01-15 数据行数: {count} {'(正确)' if count == 3 else '(数据重复!)'})")

print("\n" + "=" * 60)
print("幂等 ETL (DELETE + INSERT)")
print("=" * 60)

db_good = MockDatabase()
# 先写入其他日期的数据
db_good.idempotent_write("2024-01-14", fetch_sales_mock("2024-01-14"))

for run_num in range(1, 4):
    db_good.idempotent_write("2024-01-15", fetch_sales_mock("2024-01-15"))
    count_15 = db_good.count("2024-01-15")
    count_14 = db_good.count("2024-01-14")
    print(f"第 {run_num} 次运行后: 2024-01-15={count_15}行 {'(正确)' if count_15 == 3 else '(错误)'}, "
          f"2024-01-14={count_14}行 {'(未受影响)' if count_14 == 3 else '(受影响!)'}")

print("\n修复方案总结:")
print("  1. SQL: DELETE FROM daily_product_sales WHERE date = ? 再 INSERT")
print("  2. BigQuery: INSERT OVERWRITE TABLE ... PARTITION(date=?)")
print("  3. Snowflake/Redshift: MERGE INTO ... ON (date AND product_id)")

### 练习 3: Schema Evolution 决策

判断以下 Schema 变更是否安全，并给出处理方案：

1. 将 `amount` 列从 `FLOAT` 改为 `DECIMAL(18,4)`
2. 新增 `updated_at TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP`
3. 将 `user_id` 从 `INT` 改为 `VARCHAR(36)`（迁移到 UUID）
4. 删除已弃用 2 年的 `legacy_ref` 列
5. 将 `price` 列的含义从「美元」改为「分为单位的整数」

In [ ]:
# 练习 3 答案
schema_change_analysis = [
    {
        "变更": "FLOAT → DECIMAL(18,4)",
        "安全": "取决于场景",
        "分析": "类型变更，破坏性。但实际上精度提升是合理需求。",
        "方案": [
            "1. 新增列: amount_decimal DECIMAL(18,4) DEFAULT NULL",
            "2. 双写: 同时写 amount 和 amount_decimal",
            "3. 迁移消费端使用 amount_decimal",
            "4. 移除旧列 amount（下一个大版本）",
            "注意: FLOAT 有精度问题，迁移时需要 ROUND(amount::float, 4)"
        ]
    },
    {
        "变更": "新增 updated_at TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP",
        "安全": "安全（向后兼容）",
        "分析": "新增列有默认值，旧数据自动填充，旧代码不受影响。",
        "方案": ["直接 ALTER TABLE ADD COLUMN updated_at TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP"]
    },
    {
        "变更": "user_id: INT → VARCHAR(36) (迁移到 UUID)",
        "安全": "危险（破坏性）",
        "分析": "类型和语义同时变更，所有引用 user_id 的代码都需修改。",
        "方案": [
            "1. 新增列: user_uuid VARCHAR(36) DEFAULT NULL",
            "2. 生成 UUID 并填充历史数据",
            "3. 更新写入端，同时维护 user_id 和 user_uuid",
            "4. 逐步迁移下游，用 user_uuid 替代 user_id",
            "5. 移除 user_id（确认无引用后）",
            "这个过程可能需要 3-6 个月"
        ]
    },
    {
        "变更": "删除已弃用 2 年的 legacy_ref 列",
        "安全": "需要验证后才安全",
        "分析": "理论上已弃用，但需确认没有任何代码仍在引用。",
        "方案": [
            "1. 在 Schema Registry 标记为 deprecated",
            "2. 搜索代码库确认无引用（grep / 全局搜索）",
            "3. 检查数据血缘系统，确认无下游依赖",
            "4. 先置为 NULL（软删除），观察一个月",
            "5. 确认无报警后，再物理删除"
        ]
    },
    {
        "变更": "price 列语义从美元改为分（整数）",
        "安全": "极其危险！",
        "分析": "数据类型可能不变，但语义变了。所有下游计算 price 的逻辑都会出错。这是最危险的变更类型！",
        "方案": [
            "1. 绝对不要修改现有 price 列的语义!",
            "2. 新增列: price_cents BIGINT（明确命名单位）",
            "3. 在 ETL 中: price_cents = ROUND(price * 100)",
            "4. 所有新代码使用 price_cents",
            "5. 旧列 price 保留但标记为 deprecated"
        ]
    }
]

for item in schema_change_analysis:
    print(f"\n{'='*60}")
    print(f"变更: {item['变更']}")
    print(f"安全性: {item['安全']}")
    print(f"分析: {item['分析']}")
    print("处理方案:")
    for s in item["方案"]:
        print(f"  {s}")

### 练习 4: 数据血缘查询

基于本节创建的 `LineageGraph`，回答以下问题并用代码实现：

1. 如果 `staging.user_events` 表数据损坏，哪些报表会受影响？
2. `dashboard.sales_kpi` 数据异常，最可能的数据源根因是哪个？
3. 如何扩展 LineageGraph 支持列级血缘追踪？

In [ ]:
# 练习 4 答案
print("问题 1: staging.user_events 损坏，下游影响分析")
print("=" * 60)
downstream_of_events = lineage.get_downstream("staging.user_events", depth=5)
for ds in downstream_of_events:
    node_info = lineage.nodes.get(ds, {})
    print(f"  受影响: {ds} [{node_info.get('layer', '?')}]")

print("\n问题 2: dashboard.sales_kpi 异常，根因追溯")
print("=" * 60)
upstream_of_kpi = lineage.get_upstream("dashboard.sales_kpi", depth=10)
sources = [ds for ds in upstream_of_kpi
           if lineage.nodes.get(ds, {}).get("layer") == "raw"]
print("  所有上游数据集:")
for ds in upstream_of_kpi:
    layer = lineage.nodes.get(ds, {}).get("layer", "?")
    print(f"    {ds} [{layer}]")
print(f"\n  最终数据源（优先排查）: {sources}")

# 问题 3: 列级血缘扩展
print("\n问题 3: 列级血缘追踪扩展")
print("=" * 60)

class ColumnLevelLineage:
    """列级血缘追踪"""
    def __init__(self):
        self.column_lineage = {}  # {"table.column": ["source_table.source_col", ...]}

    def add_column_lineage(self, target_table: str, target_col: str,
                           source_table: str, source_col: str, transform: str = None):
        key = f"{target_table}.{target_col}"
        if key not in self.column_lineage:
            self.column_lineage[key] = []
        self.column_lineage[key].append({
            "source": f"{source_table}.{source_col}",
            "transform": transform or "direct copy"
        })

    def get_column_upstream(self, table: str, column: str) -> list:
        return self.column_lineage.get(f"{table}.{column}", [])


col_lineage = ColumnLevelLineage()

# 注册列级血缘
col_lineage.add_column_lineage(
    "dw.fact_orders", "total_revenue_usd",
    "raw.orders", "amount",
    transform="amount * exchange_rate"
)
col_lineage.add_column_lineage(
    "rpt.daily_sales", "daily_revenue",
    "dw.fact_orders", "total_revenue_usd",
    transform="SUM(total_revenue_usd)"
)
col_lineage.add_column_lineage(
    "dashboard.sales_kpi", "revenue_target_pct",
    "rpt.daily_sales", "daily_revenue",
    transform="daily_revenue / monthly_target * 100"
)

print("\n列级血缘示例: dashboard.sales_kpi.revenue_target_pct 来自哪里?")
upstream = col_lineage.get_column_upstream("dashboard.sales_kpi", "revenue_target_pct")
print(f"  直接上游: {upstream}")
print("\n  追溯路径: raw.orders.amount")
print("    → dw.fact_orders.total_revenue_usd (乘以汇率)")
print("    → rpt.daily_sales.daily_revenue (SUM)")
print("    → dashboard.sales_kpi.revenue_target_pct (除以目标)")

### 练习 5: 设计完整的数据质量监控方案

设计一个电商平台数据仓库的完整数据质量监控方案，包括：
- 各层次（raw/staging/dw/reporting）的检查策略
- 告警阈值设计
- 异常响应流程
- 与 Airflow 的集成方式

In [ ]:
# 练习 5 答案
dq_monitoring_design = {
    "数据分层质量策略": {
        "raw (原始层)": {
            "工具": "Great Expectations",
            "时机": "数据摄取后立即检查",
            "检查项": [
                "schema_validation: 字段存在且类型正确",
                "freshness: 数据时间戳在预期范围内",
                "volume_check: 行数在历史均值的 50%-200% 之间",
                "null_check: 关键字段 (PK) 不为空",
                "format_check: 日期格式、邮箱格式等",
            ],
            "失败处理": "停止管道，告警，不处理原始数据"
        },
        "staging (清洗层)": {
            "工具": "dbt tests",
            "时机": "dbt run 后自动运行",
            "检查项": [
                "unique: 主键唯一性",
                "not_null: 业务关键字段非空",
                "accepted_values: 枚举值合法性",
                "relationships: 外键关联完整性",
                "custom_sql: 业务规则（如 amount > 0）",
            ],
            "失败处理": "dbt test 失败，停止后续模型构建"
        },
        "dw (数仓层)": {
            "工具": "dbt tests + re_data/elementary",
            "时机": "模型构建后",
            "检查项": [
                "anomaly_detection: 指标异常波动（z-score > 3）",
                "completeness: 每天数据完整性（不缺日期）",
                "consistency: 跨表数据一致性",
                "recency: 数据新鲜度检查",
            ],
            "失败处理": "告警 + 标记问题数据，继续运行（不阻塞报表）"
        },
        "reporting (报表层)": {
            "工具": "Monte Carlo / Custom monitors",
            "时机": "定时（每小时）监控",
            "检查项": [
                "metric_drift: 关键指标环比波动 > 20% 告警",
                "dashboard_freshness: 报表数据不超过 1 小时",
                "golden_dataset_comparison: 与人工验证数据集对比",
            ],
            "失败处理": "Pager Duty 告警，数据团队 oncall 响应"
        }
    },
    "告警级别设计": {
        "P0 (立即响应, <15min)": "核心收入报表异常、数据完全丢失、PK 重复超过 1%",
        "P1 (4小时内响应)": "数据量异常 >50%、质量检查失败率 >5%",
        "P2 (下一工作日)": "数据量轻微异常 10-50%、非核心字段空值增加",
        "P3 (计划修复)": "文档不一致、minor 格式问题"
    },
    "Airflow 集成": '''
# Airflow DAG 中集成 GE + dbt tests
@dag(schedule="@daily", ...)
def data_quality_pipeline():

    @task
    def run_ge_checkpoint() -> dict:
        # 运行 Great Expectations
        result = checkpoint.run()
        if not result["success"]:
            raise ValueError("数据质量检查失败")
        return result["statistics"]

    @task
    def run_dbt_models():
        subprocess.run(["dbt", "run", "--select", "tag:daily"])

    @task
    def run_dbt_tests():
        result = subprocess.run(
            ["dbt", "test", "--select", "tag:daily"],
            capture_output=True
        )
        if result.returncode != 0:
            raise ValueError(f"dbt tests 失败: {result.stdout}")

    ge_result = run_ge_checkpoint()
    dbt_run = run_dbt_models()
    ge_result >> dbt_run  # GE 通过才运行 dbt
    run_dbt_tests()
'''
}

import json
for section, content in dq_monitoring_design.items():
    print(f"\n{'='*60}")
    print(f"{section}")
    print("=" * 60)
    if isinstance(content, dict):
        for key, value in content.items():
            print(f"\n[{key}]")
            if isinstance(value, dict):
                for k, v in value.items():
                    if isinstance(v, list):
                        print(f"  {k}:")
                        for item in v:
                            print(f"    - {item}")
                    else:
                        print(f"  {k}: {v}")
            else:
                print(f"  {value}")
    else:
        print(content)